Select a T4 runtime. Upload the repository ZIP or the fixture ZIP described in the SASS README. Both runs use the included standalone CUBIN files.

In [ ]:
from pathlib import Path
from google.colab import files
import io
import subprocess
import sys
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one ZIP containing the sass directory.")
root = Path("/content/cu-sass-test")
root.mkdir(exist_ok=True)
requested = {
    "t4/oracle.py", "t4/oracle-input.json", "bench.py",
    *[f"fixtures/build/{name}.sm75.cubin" for name in ("vectorAdd", "tripleSum", "elementMultiply", "sameNameMultiply")],
}
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for item in sorted(requested):
        matches = [name for name in archive.namelist() if name.endswith("/sass/" + item) or name == "sass/" + item]
        if len(matches) != 1:
            raise ValueError(f"Expected one copy of sass/{item}")
        (root / Path(item).name).write_bytes(archive.read(matches[0]))
subprocess.run([sys.executable, str(root / "oracle.py"), str(root)], check=True)
files.download(str(root / "sass-sm75-t4.json"))


In [ ]:
timing = root / "t4-speed.json"
subprocess.run([sys.executable, str(root / "bench.py"), "libcuda.so.1", str(root / "vectorAdd.sm75.cubin"), str(timing)], check=True)
files.download(str(timing))
